# Emotia AI: GoEmotions RoBERTa Fine-Tuning
This notebook allows you to fine-tune the `roberta-base-go_emotions` model on the Google GoEmotions multi-label dataset using Google Colab's GPU acceleration (e.g. Free T4 GPU).

### Workflow Steps:
1. **Setup GPU**: Go to `Runtime` -> `Change runtime type` -> select `GPU` (T4).
2. **Upload Files**: Upload `go_emotions_train.csv`, `go_emotions_validation.csv`, `go_emotions_test.csv`, and `preprocess.py` to your Colab session files (left-side panel).
3. **Run All Cells**: Execute this notebook cell-by-cell or select `Runtime` -> `Run all`.
4. **Download Model**: Download the zipped model folder generated at the end and extract it into your local `emotion-detection/` folder.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate pandas numpy torch scikit-learn

## 2. Verify Preprocessing module and Datasets exist

In [ ]:
import os
required_files = ["preprocess.py", "go_emotions_train.csv", "go_emotions_validation.csv", "go_emotions_test.csv"]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Please upload the following files before continuing: {missing}")
else:
    print("All required files are uploaded and verified!")

## 3. Define Training Pipeline

In [ ]:
import ast
import json
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from preprocess import normalize_text

MODEL_NAME = "SamLowe/roberta-base-go_emotions"
MODEL_FOLDER = "emotion_model"
NUM_CLASSES = 28

class GoEmotionsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

    def __len__(self):
        return len(self.labels)

class MultiLabelTrainer(Trainer):
    def __init__(self, *args, pos_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if pos_weights is not None:
            self.pos_weights = torch.tensor(pos_weights, dtype=torch.float32)
        else:
            self.pos_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        
        if self.pos_weights is not None:
            device = logits.device
            loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weights.to(device))
        else:
            loss_fn = torch.nn.BCEWithLogitsLoss()
            
        loss = loss_fn(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

def load_and_preprocess_data(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Cleaning {len(df)} texts...")
    texts = df['text'].apply(normalize_text).tolist()
    
    labels = []
    for labels_str in df['labels']:
        cleaned = labels_str.strip().replace('[', '').replace(']', '').replace(',', ' ')
        label_list = [int(x) for x in cleaned.split()]
        multi_hot = np.zeros(NUM_CLASSES, dtype=np.float32)
        for l in label_list:
            multi_hot[l] = 1.0
        labels.append(multi_hot)
        
    return texts, np.array(labels)

## 4. Run Model Training (GPU Fine-tuning)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")
if device == "cpu":
    print("WARNING: GPU is not enabled! Head to Runtime -> Change runtime type and enable GPU first.")

# Load datasets
train_texts, train_labels = load_and_preprocess_data("go_emotions_train.csv")
val_texts, val_labels = load_and_preprocess_data("go_emotions_validation.csv")

# Compute class weights
pos_counts = np.sum(train_labels, axis=0)
neg_counts = len(train_labels) - pos_counts
pos_weights = neg_counts / (pos_counts + 10)
pos_weights = np.clip(pos_weights, 1.0, 15.0)

# Initialize model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, padding=True, truncation=True, max_length=128)
val_encodings = tokenizer(val_texts, padding=True, truncation=True, max_length=128)

train_dataset = GoEmotionsDataset(train_encodings, train_labels)
val_dataset = GoEmotionsDataset(val_encodings, val_labels)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3 if device == "cuda" else 1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none"
)

trainer = MultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    pos_weights=pos_weights
)

trainer.train()

trainer.model.save_pretrained(MODEL_FOLDER)
tokenizer.save_pretrained(MODEL_FOLDER)
print("Model saved to 'emotion_model/'!")

## 5. Perform Validation Threshold Tuning

In [ ]:
print("Starting validation threshold tuning...")
model.to(device)
model.eval()

val_probs = []
eval_batch_size = 32
for i in range(0, len(val_texts), eval_batch_size):
    batch_texts = val_texts[i:i+eval_batch_size]
    inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        val_probs.append(probs)
        
val_probs = np.concatenate(val_probs, axis=0)

tuned_thresholds = []
id2label = model.config.id2label

print("\nOptimal Per-Class Thresholds Found:")
print("-" * 55)
print(f"{'Class ID':<8} | {'Emotion':<15} | {'Optimal Threshold':<18} | {'F1-Score':<8}")
print("-" * 55)

for c in range(NUM_CLASSES):
    label_name = id2label[str(c)]
    best_thresh = 0.3
    best_f1 = -1.0
    
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (val_probs[:, c] >= t).astype(int)
        true = val_labels[:, c]
        
        tp = np.sum((preds == 1) & (true == 1))
        fp = np.sum((preds == 1) & (true == 0))
        fn = np.sum((preds == 0) & (true == 1))
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
            
    if best_f1 <= 0.0:
        best_thresh = 0.3
        
    tuned_thresholds.append(float(best_thresh))
    print(f"{c:<8} | {label_name:<15} | {best_thresh:<18.2f} | {max(0.0, best_f1):<8.4f}")

print("-" * 55)

threshold_path = os.path.join(MODEL_FOLDER, "thresholds.json")
with open(threshold_path, "w") as f:
    json.dump(tuned_thresholds, f, indent=4)
print(f"Saved per-class thresholds to {threshold_path}!")

## 6. Compress and Download Model Weights

In [ ]:
import shutil
from google.colab import files

# Zip the model folder
shutil.make_archive("emotion_model", 'zip', "emotion_model")
print("Model weights compressed successfully!")

# Download the zip
files.download("emotion_model.zip")
print("Download trigger sent! Download 'emotion_model.zip' and extract it into your project folder.")